<a href="https://colab.research.google.com/github/JotaBlanco/TheValley/blob/main/EDA/04-probabilidad/06_1___Explicaci%C3%B3n_Statistics_for_Hackers_Sin_Resolver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
sns.set()

In [ ]:
import random

# 01 - Técnicas de Simulación Estadística: Inspirado en ['Statistics for Hackers' de Jake VanderPlas](https://speakerdeck.com/jakevdp/statistics-for-hackers)

Este cuaderno explora técnicas de simulación estadística como herramienta práctica para comprender conceptos complejos mediante ejemplos sencillos y reproducibles.

## 01.01 Simulación Directa

La **Simulación Directa** es una técnica computacional que nos permite modelar y analizar el comportamiento de sistemas complejos o eventos aleatorios generando datos aleatorios de acuerdo con un modelo probabilístico.

**¿Cuándo es útil?**
*   Para estimar probabilidades de eventos complejos.
*   Para analizar el rendimiento de sistemas donde hay incertidumbre.
*   Para probar la robustez de un modelo bajo diferentes escenarios.
*   Cuando no se dispone de datos reales o son difíciles de obtener.

**¿Cómo funciona?**
El proceso generalmente implica los siguientes pasos:
1.  **Definir el sistema o evento:** Clarificar qué se quiere simular y cuáles son las variables clave.
2.  **Establecer un modelo probabilístico:** Determinar las distribuciones de probabilidad que rigen el comportamiento de las variables aleatorias.
3.  **Generar números aleatorios:** Utilizar generadores de números aleatorios para simular los eventos según las distribuciones definidas.
4.  **Repetir la simulación:** Realizar un gran número de ensayos para obtener una muestra representativa del comportamiento del sistema.
5.  **Analizar los resultados:** Calcular estadísticas, promedios, probabilidades y otras métricas de interés a partir de los datos simulados para sacar conclusiones.

### Ejemplo: Lanzamiento de dos dados que sumen 7

In [ ]:
# Dado
random.randint(1, 6)

In [ ]:
# Definimos el número de simulaciones o lanzamientos de dados
num_simulaciones = 100000

# Inicializamos un contador para el evento de interés
# (suma de los dados igual a 7)
contador_suma_7 = 0

# Realizamos la simulación
for _ in range(num_simulaciones):
    # Lanzar el primer dado (número aleatorio entre 1 y 6)
    dado1 = random.randint(1, 6)
    # Lanzar el segundo dado (número aleatorio entre 1 y 6)
    dado2 = random.randint(1, 6)

    # Calcular la suma de los resultados
    suma_dados = dado1 + dado2

    # Comprobar si la suma es igual a 7 y, si lo es, incrementar el contador
    if suma_dados == 7:
        contador_suma_7 += 1

In [ ]:
# --- Análisis de los resultados ---

# Calculamos la probabilidad estimada del evento (suma igual a 7)
probabilidad_estimada = contador_suma_7 / num_simulaciones

# Imprimimos los resultados
print(f"Número de veces que la suma fue 7: {contador_suma_7}")
print(f"Probabilidad estimada de que la suma sea 7: {probabilidad_estimada:.4f}")

# Nota: La probabilidad teórica de que la suma de dos dados sea 7 es:
# 6/36 = 0.1667


### EJERICIO: Baraja
Probabilidad de, si te reparten 4 cartas de la baraja, obtener una pareja.

In [ ]:
baraja = [1,2,3,4,5,6,7,8,9,10] * 4
np.random.choice(baraja, size=4, replace=False)

### Ejemplo: Generalización de la simulación

In [ ]:
sim_dict = {
    "Dado 1": random.randint(1, 6),
    "Dado 2": random.randint(1, 6)
    }
pd.DataFrame([sim_dict])

In [ ]:
def sim_dados(n_dados):
    resultados_dados = {}
    for i in range(n_dados):
        resultados_dados[f"Dado {i+1}"] = random.randint(1, 6)
    return pd.DataFrame([resultados_dados])

In [ ]:
sim_dados(2)

In [ ]:
def escenario_simulacion(num_simulaciones, funcion_sim):
    df_sim = pd.DataFrame()
    for _ in range(num_simulaciones):
        df_sim_i = funcion_sim()
        df_sim = pd.concat([df_sim, df_sim_i], ignore_index=True)
    return df_sim

In [ ]:
df_dados = escenario_simulacion(
    num_simulaciones = 100000,
    funcion_sim = lambda: sim_dados(n_dados=2))
df_dados

In [ ]:
(df_dados.sum(axis=1)>10).mean()

## 01.02 Barajado (Shuffling)

El **Barajado** o _Shuffling_ es una técnica de remuestreo para pruebas de hipótesis, especialmente cuando queremos evaluar la significación de una diferencia observada. Se inspira en la idea de que, si no hay una diferencia real entre grupos o tratamientos, cualquier asignación de datos a estos grupos es igualmente probable.

**¿Por qué es útil?**
*   **Comparación de grupos:** Ideal para probar si la diferencia observada entre dos o más grupos es mayor de lo que cabría esperar por puro azar.
*   **Flexibilidad:** Funciona bien con datos de cualquier distribución y es robusto ante valores atípicos.
*   **Intuitivo:** La lógica detrás del barajado es fácil de entender: si no hay un efecto, "mezclar" los datos no debería cambiar fundamentalmente la métrica de interés.

**¿Cómo funciona?**
El proceso general para una prueba de permutación (basada en barajado) es el siguiente:
1.  **Combinar los datos:** Se combinan los datos de todos los grupos o condiciones que se desean comparar.
2.  **Generar muestras permutadas:** Se barajan aleatoriamente los datos combinados y se dividen en nuevos "grupos" del mismo tamaño que los originales. Esto simula la hipótesis nula, es decir, la idea de que los datos provienen de la misma población y cualquier diferencia es aleatoria.
3.  **Calcular el estadístico de prueba:** Para cada una de estas muestras permutadas, se calcula el estadístico de prueba de interés (por ejemplo, la diferencia de medias entre los grupos).
4.  **Construir la distribución nula:** Se repiten los pasos 2 y 3 miles de veces para construir una distribución de valores del estadístico de prueba bajo la hipótesis nula.
5.  **Calcular el valor p:** Se compara el estadístico de prueba observado en los datos originales con esta distribución nula para determinar cuántas veces el estadístico permutado fue tan extremo o más extremo que el observado. Esta proporción es el valor p empírico.

### Ejemplo: Campaña de Marketing

#### Importar dataset

In [ ]:
df = pd.read_csv("/content/marketing_AB.csv").drop(columns=["Unnamed: 0"]).iloc[:5000].reset_index(drop=True)
df.columns = ['user_id', 'test_group', 'converted', 'total_ads', 'most_ads_day', 'most_ads_hour']
df["converted"] = df["converted"].astype(int)
df

#### Efecto de la campaña real
Columna test_group

In [ ]:
df.groupby("test_group")["converted"].describe()

In [ ]:
df["test_group"].value_counts()/len(df)

In [ ]:
ad_mean = df.loc[df["test_group"]=="ad","converted"].mean()
print(ad_mean)
psa_mean = df.loc[df["test_group"]=="psa","converted"].mean()
print(psa_mean)

In [ ]:
ad_mean - psa_mean

In [ ]:
# Comparación abs: puntos porcentuales
(ad_mean - psa_mean)*100

In [ ]:
# Comparación relativa (%)
ad_mean/psa_mean

#### Función que genera diferencia

In [ ]:
def funcion_obtener_diferencia_media(df, col_vector, target_name):
    df_temp = df.copy() # Make a copy to avoid modifying the original dataframe
    df_temp["groupbycol"] = col_vector
    stats = df_temp.groupby("groupbycol")[target_name].mean()
    return float(stats["ad"] - stats["psa"])

In [ ]:
funcion_obtener_diferencia_media(
    df = df,
    col_vector = df["test_group"],
    target_name= "converted")

#### Barajado
Generamos etiquetas falsas de campaña/grupo de control.

In [ ]:
df["test_group"].sample(frac=1).reset_index(drop=True)

In [ ]:
funcion_obtener_diferencia_media(
    df = df,
    col_vector = df["test_group"].sample(frac=1).reset_index(drop=True),
    target_name = 'converted')

#### Simular muchas veces

In [ ]:
num_simulaciones = 10000

diferencias = []

for _ in range(num_simulaciones):
    diferencia_i = funcion_obtener_diferencia_media(
                      df = df,
                      col_vector = df["test_group"].sample(frac=1).reset_index(drop=True),
                      target_name = 'converted')
    diferencias.append(float(diferencia_i))

diferencias[:10]


In [ ]:
diff_real = ad_mean - psa_mean
diff_real

#### Estudiar distribución

In [ ]:
# Media
plt.figure(figsize=(10, 6))
sns.histplot(np.array(diferencias), kde=True, bins=20)
plt.axvline(diff_real, color='red', linestyle='--', label=f'Observed Difference: {diff_real:.4f}')
plt.title('Distribution of Shuffled Differences')
plt.xlabel('Difference')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Probabiliad de que nuestro efecto en la campaña fuera fruto del azar
(np.abs(diferencias)>=abs(diff_real)).mean()

### Ejemplo II: Añadimos más datos
Nuevo día/semana, recibimos más datos de la campaña.

In [ ]:
df_2 = pd.read_csv("/content/marketing_AB.csv").drop(columns=["Unnamed: 0"]).iloc[5000:10000].reset_index(drop=True)
df_2.columns = ['user_id', 'test_group', 'converted', 'total_ads', 'most_ads_day', 'most_ads_hour']
df_2["converted"] = df_2["converted"].astype(int)
df_2

In [ ]:
df.shape

In [ ]:
df_2.shape

In [ ]:
df_12 = pd.concat([df, df_2], ignore_index=True)
df_12

#### Efecto de la campaña

In [ ]:
def devuelve_efecto_medios(df):
  ad_mean = df.loc[df["test_group"]=="ad","converted"].mean()
  print("Converted medio para group ad", ad_mean)
  psa_mean = df.loc[df["test_group"]=="psa","converted"].mean()
  print("Converted medio para group psa",psa_mean)
  return float(ad_mean), float(psa_mean)

In [ ]:
devuelve_efecto_medios(df)

In [ ]:
devuelve_efecto_medios(df_2)

In [ ]:
devuelve_efecto_medios(df_12)

#### Barajar y simular

In [ ]:
num_simulaciones = 10000

diferencias_12 = []

for _ in range(num_simulaciones):
    diferencia_i = funcion_obtener_diferencia_media(
                      df = df_12,
                      col_vector = df_12["test_group"].sample(frac=1).reset_index(drop=True),
                      target_name = 'converted')
    diferencias_12.append(float(diferencia_i))

diferencias_12[:10]

#### Estudiar distribución
E hipótesis nula

In [ ]:
ad_mean_12, psa_mean_12 = devuelve_efecto_medios(df_12)
diff_real_12 = ad_mean_12 - psa_mean_12
diff_real_12

In [ ]:
# Media
plt.figure(figsize=(10, 6))
sns.histplot(np.array(diferencias_12), kde=True, bins=20)
plt.axvline(diff_real_12, color='red', linestyle='--', label=f'Observed Difference: {diff_real_12:.4f}')
plt.title('Distribution of Shuffled Differences')
plt.xlabel('Difference')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Probabiliad de que nuestro efecto en la campaña fuera fruto del azar
(np.abs(diferencias_12)>=abs(diff_real_12)).mean()

### EJERCICIO: Llegan nuevos datos
Nos llegan una nueva remesa de datos de la campaña. Actualiza nuestra seguridad sobre el efecto que está teniendo.

In [ ]:
df_3 = pd.read_csv("/content/marketing_AB.csv").drop(columns=["Unnamed: 0"]).iloc[10000:15000].reset_index(drop=True)
df_3.columns = ['user_id', 'test_group', 'converted', 'total_ads', 'most_ads_day', 'most_ads_hour']
df_3["converted"] = df_3["converted"].astype(int)
df_3

#### Efecto de la campaña
Recalcular el efecto, y estudiar la evolución.

#### Barajar y simular

#### Estudiar distribución
E hipótesis nula

### Estudiar evolución de confianza

In [ ]:
all_diferencias = np.concatenate([diferencias_123, diferencias_12, diferencias])
x_min = np.min(all_diferencias)
x_max = np.max(all_diferencias)

In [ ]:
# Media
plt.figure(figsize=(10, 6))
sns.histplot(np.array(diferencias_123), kde=True, bins=20)
plt.axvline(diff_real_123, color='red', linestyle='--', label=f'Observed Difference: {diff_real_123:.4f}')
plt.title('Distribution of Shuffled Differences')
plt.xlabel('Difference')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 7))

# Escenario 1: 'diferencias' y 'diff_real'
sns.kdeplot(np.array(diferencias), color='blue', fill=False, alpha=0.2, linewidth=1, label='Escenario 1 (diferencias)')
plt.axvline(diff_real, color='red', linestyle='--', linewidth=1, alpha=0.5, label=f'Diferencia Real 1: {diff_real:.4f}')

# Escenario 2: 'diferencias_12' y 'diff_real_12'
sns.kdeplot(np.array(diferencias_12), color='blue', fill=False, alpha=0.4, linewidth=2, label='Escenario 2 (diferencias_12)')
plt.axvline(diff_real_12, color='red', linestyle='-.', linewidth=2, alpha=0.7, label=f'Diferencia Real 2: {diff_real_12:.4f}')

# Escenario 3: 'diferencias_123' y 'diff_real_123'
sns.kdeplot(np.array(diferencias_123), color='blue', fill=False, alpha=0.6, linewidth=3, label='Escenario 3 (diferencias_123)')
plt.axvline(diff_real_123, color='red', linestyle='-', linewidth=3, label=f'Diferencia Real 3: {diff_real_123:.4f}')

plt.title('Evolución de la Distribución de Diferencias Barajadas y Diferencias Reales')
plt.xlabel('Diferencia de Medias')
plt.ylabel('Densidad')
plt.legend()
plt.grid(True)
plt.xlim(x_min, x_max) # Usar los límites calculados previamente
plt.show()

## 01.03 Bootstrapping

El **Bootstrapping** es una potente técnica de remuestreo no paramétrico que se utiliza para estimar la distribución de muestreo de un estadístico (como la media, la mediana, la desviación estándar, la correlación, etc.) o para construir intervalos de confianza y realizar pruebas de hipótesis.

**¿Qué es y para qué se utiliza?**
*   **Estimación de la distribución muestral:** Permite estimar la distribución de un estadístico mediante el remuestreo de los datos observados. Esto es crucial para entender la variabilidad de nuestras estimaciones.
*   **Construcción de intervalos de confianza:** Es una forma robusta de calcular intervalos de confianza para prácticamente cualquier estadístico, sin necesidad de asumir distribuciones paramétricas.
*   **Estimación de errores estándar:** Ayuda a estimar el error estándar de un estadístico, lo que a su vez es fundamental para la inferencia estadística.
*   **Pruebas de hipótesis:** Se puede adaptar para realizar pruebas de hipótesis cuando los métodos tradicionales no son aplicables.

**¿Cómo funciona? (Remuestreo con Reemplazo)**
El proceso básico del bootstrapping es el siguiente:
1.  **Muestra original:** Se comienza con un único conjunto de datos de tamaño `n` (la muestra original).
2.  **Remuestreo con reemplazo:** Se generan un gran número (`B`, por ejemplo, 1000 a 10000) de "muestras bootstrap" a partir de la muestra original. Cada muestra bootstrap se crea seleccionando aleatoriamente `n` observaciones de la muestra original **con reemplazo**. Esto significa que algunas observaciones de la muestra original pueden aparecer varias veces en una muestra bootstrap, mientras que otras pueden no aparecer en absoluto.
3.  **Cálculo del estadístico:** Para cada una de estas `B` muestras bootstrap, se calcula el estadístico de interés (por ejemplo, la media) y se almacena el resultado.
4.  **Distribución bootstrap:** Al final, se tiene una colección de `B` valores del estadístico de interés, que forman la "distribución bootstrap" del estadístico. Esta distribución nos proporciona información sobre la forma, el centro y la dispersión del estadístico.
5.  **Inferencia:** A partir de la distribución bootstrap, se pueden estimar el error estándar del estadístico, calcular intervalos de confianza percentiles (por ejemplo, el intervalo entre el percentil 2.5 y 97.5 para un IC del 95%), o incluso realizar pruebas de hipótesis.

**Ventajas:**
*   **No paramétrico:** No requiere suposiciones sobre la distribución subyacente de la población (a diferencia de las pruebas t o Z).
*   **Versátil:** Puede aplicarse a una amplia gama de estadísticos y modelos complejos.
*   **Sencillo de implementar:** El concepto es relativamente fácil de entender y aplicar computacionalmente.
*   **Funciona con muestras pequeñas:** Útil cuando las muestras son demasiado pequeñas para la inferencia estadística tradicional.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Ejemplo de Bootstrapping: Estimación de Intervalo de Confianza para la Media ---

# 1. Generar un conjunto de datos de ejemplo
# Supongamos que tenemos una muestra de datos de mediciones
# (por ejemplo, el tiempo de respuesta de un servidor en milisegundos)
np.random.seed(42) # Para reproducibilidad
datos_originales = np.concatenate([
    np.random.normal(loc=100, scale=10, size=50), # Grupo 1
    np.random.normal(loc=110, scale=15, size=30)  # Grupo 2 (para hacer la distribución no perfectamente normal)
])

n_original = len(datos_originales)
print(f"Tamaño de la muestra original: {n_original}")
print(f"Media de la muestra original: {np.mean(datos_originales):.2f}\n")

# 2. Definir el número de remuestreos (muestras bootstrap)
num_muestras_bootstrap = 10000

# Lista para almacenar las medias de cada muestra bootstrap
medias_bootstrap = []

# 3. Realizar el remuestreo con reemplazo y calcular el estadístico de interés (media)
for _ in range(num_muestras_bootstrap):
    # Crear una muestra bootstrap seleccionando n_original elementos con reemplazo
    muestra_bootstrap = np.random.choice(datos_originales, size=n_original, replace=True)

    # Calcular la media de esta muestra bootstrap y almacenarla
    media_bootstrap = np.mean(muestra_bootstrap)
    medias_bootstrap.append(media_bootstrap)

# Convertir la lista a un array de numpy para facilitar cálculos
medias_bootstrap = np.array(medias_bootstrap)

# 4. Utilizar la distribución de estas medias bootstrap para calcular un intervalo de confianza

# Queremos un Intervalo de Confianza del 95% (percentil 2.5 y percentil 97.5)
limite_inferior_ic = np.percentile(medias_bootstrap, 2.5)
limite_superior_ic = np.percentile(medias_bootstrap, 97.5)

print(f"Número de muestras bootstrap realizadas: {num_muestras_bootstrap}")
print(f"Media de la distribución bootstrap: {np.mean(medias_bootstrap):.2f}")
print(f"Error estándar de la media (estimado por bootstrap): {np.std(medias_bootstrap):.2f}")
print(f"Intervalo de Confianza del 95% para la media: [{limite_inferior_ic:.2f}, {limite_superior_ic:.2f}]")

# 5. Visualizar la distribución bootstrap de la media
plt.figure(figsize=(10, 6))
plt.hist(medias_bootstrap, bins=50, density=True, alpha=0.7, color='lightgreen', edgecolor='black')
plt.axvline(np.mean(datos_originales), color='red', linestyle='dashed', linewidth=2, label='Media Original')
plt.axvline(limite_inferior_ic, color='blue', linestyle='dotted', linewidth=2, label='Límite Inferior IC 95%')
plt.axvline(limite_superior_ic, color='blue', linestyle='dotted', linewidth=2, label='Límite Superior IC 95%')
plt.title('Distribución Bootstrap de la Media')
plt.xlabel('Valor de la Media')
plt.ylabel('Densidad')
plt.legend()
plt.grid(axis='y', alpha=0.75)
plt.show()